<a href="https://colab.research.google.com/github/LauraaRoman/PhishAid/blob/main/phishaid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch pandas gradio

### Data Synchronization: Integrating the External Knowledge Base
This section is critical for the modularity and scalability of the PhishAid project. By fetching the remedies.csv file directly from our GitHub repository, we implement a Data-Driven Architecture.

Why we use this approach:

Decoupled Logic: The AI engine (the code) is separated from the remediation data (the CSV). This allows the Data Architect to update security protocols on GitHub without needing to modify or redeploy the source code.

Real-time Collaboration: My teammate can refine the remediation steps in the CSV file independently. The next time the script runs, it will automatically pull the most recent version using the !wget command.

Dynamic Knowledge Base: This ensures that PhishAid remains relevant as new phishing techniques emerge, simply by expanding the external dataset.

In [3]:
!wget -O remedies.csv https://raw.githubusercontent.com/LauraaRoman/PhishAid/refs/heads/main/remedies.csv
import pandas as pd
try:
    remedies_df = pd.read_csv('remedies.csv')
    print(f"System: Successfully synchronized PhishAid with the GitHub database ({len(remedies_df)} rules loaded).")
except Exception as e:
    print(f"Error: Could not load the CSV file. Please check the raw URL. Details: {e}")

--2026-05-16 15:22:08--  https://raw.githubusercontent.com/LauraaRoman/PhishAid/refs/heads/main/remedies.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1646 (1.6K) [text/plain]
Saving to: ‘remedies.csv’

remedies.csv        100%[===================>]   1.61K  --.-KB/s    in 0s      

2026-05-16 15:22:08 (27.7 MB/s) - ‘remedies.csv’ saved [1646/1646]

System: Successfully synchronized PhishAid with the GitHub database (14 rules loaded).


### AI Model Integration: Multi-Task Learning
In this stage, we connect our application to the HuggingFace Hub to load two specialized pre-trained models. This approach allows PhishAid to perform multi-dimensional analysis on any suspicious message.

Tasks identified for this project:

Phishing Detection (Text Classification): We use a BERT-based model fine-tuned specifically to distinguish between legitimate communications and phishing attempts.

Romanian Sentiment Analysis: To enhance detection, we integrate a model from ReaderBench that analyzes the tone of the message in Romanian. This helps identify "Social Engineering" tactics, such as creating a sense of urgency or fear to manipulate the user.

This dual-model setup ensures a higher level of accuracy and provides contextual information for the final security report.

In [4]:
from transformers import pipeline
phish_detector = pipeline("text-classification", model="ealvaradob/bert-finetuned-phishing")
sentiment_analyser = pipeline("text-classification", model="readerbench/ro-sentiment")

print("Success: AI Models have been successfully downloaded and loaded into memory.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/845 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/460M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Success: AI Models have been successfully downloaded and loaded into memory.


###Core Logic: Translating AI Predictions into Remediation
This is the "brain" of the application. The engine takes the raw predictions from the AI models and correlates them with our dynamic knowledge base (remedies.csv).

The Decision Workflow:

1. Data Ingestion: The engine accepts a message from the user.

2. AI Processing: It runs the text through both the Phishing and Sentiment models.

3. Contextual Mapping: If a threat is detected, it performs a keyword search within the CSV file to find the specific "First Aid" steps required for that category (e.g., Banking, Social Media).

4. Report Generation: It outputs a human-readable safety report, fulfilling the project's goal of being a "Translator" of cyber threats.

In [5]:
from datetime import datetime
import os
import csv

def phishaid_engine(message, history, visible_log):
    phish_res = phish_detector(message)[0]
    tone_res = sentiment_analyser(message)[0]

    prediction_label = phish_res['label'].lower()
    confidence_score = phish_res['score']

    tone_mapping = {
        "LABEL_0": "Urgent / Negative (Panic-based)",
        "LABEL_1": "Neutral / Informative",
        "LABEL_2": "Positive / Friendly"
    }
    friendly_tone = tone_mapping.get(tone_res['label'], "Undefined")

    report = f"🐠 PHISHAID SECURITY DIAGNOSIS \n"
    report += f"Classification: {prediction_label.upper()} (Confidence: {confidence_score:.2f})\n"
    report += f"Message Tone: {friendly_tone}\n\n"
    verdict = ""

    if prediction_label in ["phishing", "label_1"]:
        verdict = "PHISHING"
        report += "ALERT: THIS MESSAGE IS HIGHLY SUSPICIOUS!\n"
        report += "EMERGENCY ACTION PLAN (from GitHub Database):\n"

        found_actions = []
        for index, row in remedies_df.iterrows():
            if str(row['Keyword']).lower() in message.lower():
                found_actions.append(f"- [{row['Category']}]: {row['Action_Plan']}")

        if found_actions:
            report += "\n".join(found_actions)
        else:
            report += "- General Security Alert: Avoid clicking links or sharing credentials."
    else:
        report += "STATUS: This message appears to be safe. No immediate action required."

    file_name = "chat_logs.csv"
    crt_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    file_exists = os.path.isfile(file_name)

    with open(file_name, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Timestamp", "Message", "Verdict", "Confidence", "Tone"])
        writer.writerow([crt_time, message, prediction_label.upper(), f"{confidence_score:.2f}", friendly_tone])

    history.append((message, report))
    log_msg = (message[:30] + '...') if len(message) > 30 else message
    new_entry = pd.DataFrame([[log_msg, verdict]], columns=["Mesaj Scanat", "Verdict"])
    visible_log = pd.concat([visible_log, new_entry], ignore_index=True)

    return "", history, visible_log

###Verification: Testing with Real-World Scenarios
To ensure the integrity of our system, we perform an automated test using scenarios that include keywords defined in our remedies.csv. This validates that the engine correctly identifies the threat and retrieves the corresponding "First Aid" steps from the database. This section serves as our Prediction Analysis for project evaluation.

In [8]:
print("Available Keywords in CSV:", remedies_df['Keyword'].tolist())
print("-" * 50)

test_scenarios = [
    "Your bank card has been flagged for suspicious activity. Please verify your identity.",
    "Your social media password was changed. If this wasn't you, click here.",
    "URGENT: Download this security file to protect your computer immediately!",
]

simulated_chat_history = []
simulated_visible_log = pd.DataFrame(columns=["Message Scanned", "Verdict"])

for i, scenario in enumerate(test_scenarios):
    print(f"\n Running test {i+1}: {scenario}")

    _, simulated_chat_history, simulated_visible_log = phishaid_engine(
        scenario,
        simulated_chat_history,
        simulated_visible_log
    )

    last_report = simulated_chat_history[-1][1]
    print(last_report)
    print("-" * 50)

print(simulated_visible_log)


Available Keywords in CSV: ['card', 'transfer', 'iban', 'identity', 'password', 'instagram', 'hacked', 'wallet', 'airdrop', 'delivery', 'parcel', 'document', 'file', 'immediately']
--------------------------------------------------

 Running test 1: Your bank card has been flagged for suspicious activity. Please verify your identity.
🐠 PHISHAID SECURITY DIAGNOSIS 
Classification: BENIGN (Confidence: 1.00)
Message Tone: Urgent / Negative (Panic-based)

STATUS: This message appears to be safe. No immediate action required.
--------------------------------------------------

 Running test 2: Your social media password was changed. If this wasn't you, click here.
🐠 PHISHAID SECURITY DIAGNOSIS 
Classification: PHISHING (Confidence: 1.00)
Message Tone: Neutral / Informative

ALERT: THIS MESSAGE IS HIGHLY SUSPICIOUS!
EMERGENCY ACTION PLAN (from GitHub Database):
- [SocialMedia]: Reset your credentials, enable 2FA, and log out from all devices.
-------------------------------------------------

Gradio User Interface


In [9]:
import gradio as gr

with gr.Blocks() as demo:
    gr.HTML("<h1 id='title'>🐠 PhishAid Security Dashboard 𓆝 ⋆｡𖦹°‧</h1>")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🗂️ Message History")
            history_table = gr.Dataframe(
                headers=["Message Scaned", "Verdict"],
                datatype=["str", "str"],
                interactive=False,
                value=pd.DataFrame(columns=["Messaged Scanned", "Verdict"]) # Structură inițială goală
            )

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, label="PhishAid Assistant")

            with gr.Row():
                msg_input = gr.Textbox(show_label=False, placeholder="Copy suspicious message here...", scale=4)
                send_btn = gr.Button("Scan message", variant="primary", scale=1)

    msg_input.submit(
        fn=phishaid_engine,
        inputs=[msg_input, chatbot, history_table],
        outputs=[msg_input, chatbot, history_table]
    )
    send_btn.click(
        fn=phishaid_engine,
        inputs=[msg_input, chatbot, history_table],
        outputs=[msg_input, chatbot, history_table]
    )

if __name__ == "__main__":
    demo.launch()


/tmp/ipykernel_7279/3435089985.py:17: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500, label="PhishAid Assistant")
/tmp/ipykernel_7279/3435089985.py:17: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500, label="PhishAid Assistant")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c6b60a169aae7dfa88.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
